In [1]:
import pandas as pd
import numpy as np

test = pd.read_csv("test.csv")
test.head()
test.shape

(1459, 80)

In [2]:
train= pd.read_csv("train.csv")
train.head()
train.shape

(1460, 81)

In [5]:
## concat both and fill NaN 

In [6]:
test["SalePrice"] = -1
data = pd.concat([train, test], ignore_index=True)

In [29]:
mode_cols = ["SaleType", "Exterior1st", "Exterior2nd", "Electrical", "KitchenQual", "Functional", "Utilities", "MSZoning"]

none_cols = ["PoolQC", "Alley", "GarageType", "GarageFinish", "GarageQual", "GarageCond","MiscFeature", "Fence", "FireplaceQu", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]

In [30]:
data[none_cols] = data[none_cols].fillna("None")
data[mode_cols] = data[mode_cols].fillna(data[mode_cols].mode().iloc[0])

In [31]:
data.loc[data["TotalBsmtSF"] == 0,
         ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
          "BsmtFullBath", "BsmtHalfBath"]] = 0

In [32]:
data.loc[data["BsmtQual"] == "None" ,
         ["BsmtFinSF1","TotalBsmtSF", "BsmtFinSF2", "BsmtUnfSF",
          "BsmtFullBath", "BsmtHalfBath"]] = 0

In [33]:
data.loc[data["GarageType"] == "None",
         ["GarageArea","GarageCars"]] = 0

In [34]:
((data["MasVnrType"].isnull()) & (data["MasVnrArea"].isnull())).sum()

np.int64(23)

In [35]:
mask = data["MasVnrArea"].isnull() | (data["MasVnrArea"] == 0)
data.loc[mask, "MasVnrArea"] = 0
data.loc[mask, "MasVnrType"] = "None"

In [36]:
data.loc[
    data["MasVnrType"].isnull() & (data["MasVnrArea"] > 0),
    "MasVnrType"
] = data["MasVnrType"].mode()[0]

In [37]:
((data["GarageType"] == "None") & (data["GarageArea"].isnull())).sum()

np.int64(0)

In [38]:
data[data["GarageCars"].isnull()][["GarageType", "GarageArea", "GarageCars"]]

,GarageType,GarageArea,GarageCars
2576,Detchd,NaN,NaN


In [39]:
data["GarageCars"] = data["GarageCars"].fillna(data["GarageCars"].median())
data["GarageArea"] = data["GarageArea"].fillna(data["GarageArea"].median())

In [40]:
data["LotFrontage"] = data["LotFrontage"].fillna(data["LotFrontage"].median())

In [41]:
data.loc[data["GarageType"] == "None", "GarageYrBlt"] = 0

In [42]:
data["GarageYrBlt"] = data["GarageYrBlt"].fillna(data["GarageYrBlt"].median())

In [43]:
missing = data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)
## missing evaluated

Series([], dtype: int64)


In [44]:
data["TotalSF"] = data["TotalBsmtSF"] + data["1stFlrSF"] + data["2ndFlrSF"]

data["TotalBath"] = data["FullBath"] + 0.5*data["HalfBath"] + data["BsmtFullBath"] + 0.5*data["BsmtHalfBath"]

data["HouseAge"] = data["YrSold"] - data["YearBuilt"]

data["RemodAge"] = data["YrSold"] - data["YearRemodAdd"]

data["TotalPorchSF"] = data["OpenPorchSF"] + data["EnclosedPorch"] + data["3SsnPorch"] + data["ScreenPorch"]

In [ ]:
## Handling categorial features

In [49]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [50]:
ordinal_cols = ["LotShape", "Utilities", "LandSlope", "ExterQual", "ExterCond", "BsmtQual","Fence", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "HeatingQC", "KitchenQual", "Functional", "FireplaceQu", "GarageFinish", "GarageQual", "GarageCond", "PavedDrive", "PoolQC"]

nominal_cols = ["MSZoning", "Street", "Alley", "LandContour", "LotConfig", "Neighborhood","MiscFeature", "Condition1", "Condition2", "BldgType", "HouseStyle", "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd", "MasVnrType", "Foundation", "Heating", "CentralAir", "Electrical", "GarageType", "SaleType", "SaleCondition"]

In [51]:
ordinal_encoder = OrdinalEncoder()
ohe_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
data[ordinal_cols] = ordinal_encoder.fit_transform(data[ordinal_cols])

encoded = ohe_encoder.fit_transform(data[nominal_cols])
encoded = pd.DataFrame(
    encoded,
    columns=ohe_encoder.get_feature_names_out(nominal_cols),
    index=data.index
)

data = data.drop(columns=nominal_cols)
data = pd.concat([data, encoded], axis=1)

In [67]:
data.drop("Id", axis=1, inplace=True)

In [52]:
data.shape

(2919, 232)

In [89]:
X = X.drop(columns=zero_features)
df2 = df2.drop(columns=zero_features)

In [90]:
df1 = data[data["SalePrice"] != -1].copy()
df2 = data[data["SalePrice"] == -1].drop(columns="SalePrice").copy()

X = df1.drop(columns="SalePrice")
y = np.log1p(df1["SalePrice"])

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split( X, y, test_size=0.2, random_state=42)
## Not my idea but to calculate accuracy this is best

In [91]:
X_train.shape

(1168, 230)

In [92]:
## numerical vals scaling

In [93]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

In [94]:
print(num_cols)

Index(['MSSubClass', 'LotFrontage', 'LotArea', 'LotShape', 'Utilities',
       'LandSlope', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       ...
       'SaleType_ConLw', 'SaleType_New', 'SaleType_Oth', 'SaleType_WD',
       'SaleCondition_Abnorml', 'SaleCondition_AdjLand',
       'SaleCondition_Alloca', 'SaleCondition_Family', 'SaleCondition_Normal',
       'SaleCondition_Partial'],
      dtype='object', length=230)


In [95]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])

df2[num_cols] = scaler.transform(df2[num_cols])

In [96]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [97]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score

In [98]:
xgb = XGBRegressor(
    n_estimators=700,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.6,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)
y_pred = np.expm1(xgb.predict(X_val))
y_true = np.expm1(y_val)

print("RMSE:", np.sqrt(mean_squared_error(y_true, y_pred)))
print("MAE:", mean_absolute_error(y_true, y_pred))
print("R2:", r2_score(y_true, y_pred))

RMSE: 26459.52373446577
MAE: 15184.780474101033
R2: 0.9087253426589922


In [99]:
test_pred = np.expm1(xgb.predict(df2))
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": test_pred
})

In [100]:
submission.to_csv("submission4_features_added2.csv", index=False)

In [79]:
print(submission.head())
print(submission.columns)

     Id      SalePrice
0  1461  126264.898438
1  1462  174913.062500
2  1463  188884.671875
3  1464  196476.078125
4  1465  193465.046875
Index(['Id', 'SalePrice'], dtype='object')


In [80]:
submission.shape
submission.columns

Index(['Id', 'SalePrice'], dtype='object')

In [82]:
import pandas as pd
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance.head(20))
print(importance.tail(20))

                  Feature  Importance
6             OverallQual    0.120686
56                TotalSF    0.110987
33            KitchenQual    0.062573
40             GarageCars    0.056938
57              TotalBath    0.049607
32           KitchenAbvGr    0.044022
201          CentralAir_N    0.043380
11              ExterQual    0.035133
36             Fireplaces    0.025294
26              GrLivArea    0.019546
43             GarageCond    0.019071
209     GarageType_Attchd    0.018242
58               HouseAge    0.018204
13               BsmtQual    0.015526
127         BldgType_1Fam    0.014623
165    Exterior1st_Stucco    0.014212
9            YearRemodAdd    0.012430
86   Neighborhood_Crawfor    0.011802
65            MSZoning_RM    0.011629
71        LandContour_Bnk    0.011207
                 Feature  Importance
144    RoofStyle_Mansard         0.0
146     RoofMatl_ClyTile         0.0
107     MiscFeature_Othr         0.0
148     RoofMatl_Membran         0.0
178    Exterior2n

In [84]:
zero_features = importance[importance["Importance"] == 0]["Feature"].tolist()
print(zero_features)

['Heating_OthW', 'Electrical_FuseP', 'Utilities', 'GarageType_2Types', 'SaleCondition_Alloca', 'Foundation_Stone', 'SaleCondition_AdjLand', 'Neighborhood_NPkVill', 'SaleType_ConLw', 'SaleType_CWD', 'Neighborhood_IDOTRR', 'Heating_Wall', 'SaleType_Con', 'Heating_Floor', 'SaleType_ConLI', 'Electrical_Mix', 'Neighborhood_Blueste', 'MiscFeature_Gar2', 'RoofStyle_Shed', 'HouseStyle_SFoyer', 'HouseStyle_2.5Unf', 'HouseStyle_2.5Fin', 'Condition1_PosA', 'HouseStyle_1.5Unf', 'Condition2_RRNn', 'Condition2_RRAn', 'Condition2_RRAe', 'Condition2_PosN', 'Condition2_PosA', 'Condition2_Norm', 'Condition2_Feedr', 'Condition2_Artery', 'Condition1_RRNn', 'Condition1_RRNe', 'RoofStyle_Mansard', 'RoofMatl_ClyTile', 'MiscFeature_Othr', 'RoofMatl_Membran', 'Exterior2nd_Other', 'Exterior2nd_ImStucc', 'MiscFeature_Shed', 'Exterior2nd_CBlock', 'Street_Grvl', 'Street_Pave', 'Exterior1st_Stone', 'Exterior1st_ImStucc', 'Exterior1st_CBlock', 'MiscFeature_TenC', 'Exterior1st_AsphShn', 'RoofMatl_WdShngl', 'RoofMatl_

AttributeError: 'list' object has no attribute 'shape'